# Fashion-MNIST Baseline Experiment

Train the baseline CNN on Fashion-MNIST and log metrics.

In [4]:
import sys
from pathlib import Path
import time

candidate_roots = [
    Path('/content/ouroboros'),
]
project_root = next((p for p in candidate_roots if p.exists()), None)
if project_root is None:
    raise FileNotFoundError('Project root not found. Update candidate_roots.')
sys.path.insert(0, str(project_root))

import torch
from torch import nn
from torch.optim import Adam

from src.data_loaders import get_fashion_mnist_loaders
from src.metrics import MetricsLogger, compute_system_metrics, plot_learning_curves, reset_cuda_peak_memory
from src.models import CNN3Layer
from src.trainer import train_epoch, validate_epoch, save_checkpoint
from src.utils import get_device, set_seed, ensure_dirs

set_seed(42)
device = get_device()
ensure_dirs('results', 'results/figures', 'checkpoints')

lr = 1e-3
batch_size = 128
epochs = 5

train_loader, val_loader = get_fashion_mnist_loaders(batch_size, 2, 'assets')
model = CNN3Layer(num_classes=10, in_channels=1).to(device)
optimizer = Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

logger = MetricsLogger(run_metadata={'dataset': 'Fashion-MNIST', 'epochs': epochs})

for epoch in range(1, epochs + 1):
    reset_cuda_peak_memory()
    start = time.perf_counter()
    train_metrics = train_epoch(
        model=model,
        dataloader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        amp_enabled=(device.type == 'cuda'),
        use_compile=hasattr(torch, 'compile'),
        collect_grad_stats=True,
    )
    val_metrics = validate_epoch(
        model=model,
        dataloader=val_loader,
        criterion=criterion,
        device=device,
    )
    system_metrics = compute_system_metrics(
        total_samples=len(train_loader) * batch_size,
        start_time=start,
        end_time=time.perf_counter(),
        device=device,
    )
    logger.log_epoch(
        epoch=epoch,
        train={k: v for k, v in train_metrics.items() if k != 'gradients'},
        validation=val_metrics,
        gradients=train_metrics.get('gradients'),
        system=system_metrics,
    )
    print('Epoch', epoch, 'train', train_metrics, 'val', val_metrics)

metrics_path = Path('results') / 'fashion_mnist_baseline_metrics.json'
logger.to_json(metrics_path)
plot_learning_curves(metrics_path, output_dir='results/figures', prefix='fashion_mnist_baseline')

checkpoint_path = Path('checkpoints') / 'fashion_mnist_baseline.pth'
final_metrics = {
    'train_loss': logger.epoch_metrics[-1]['train'].get('loss', 0.0),
    'train_accuracy': logger.epoch_metrics[-1]['train'].get('accuracy', 0.0),
    'val_loss': logger.epoch_metrics[-1]['validation'].get('loss', 0.0),
    'val_accuracy': logger.epoch_metrics[-1]['validation'].get('accuracy', 0.0),
}
save_checkpoint(str(checkpoint_path), model, optimizer, epochs, final_metrics)
print('Saved metrics to', metrics_path)
print('Saved checkpoint to', checkpoint_path)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.0MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 202kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.74MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.6MB/s]
/usr/local/lib/python3.12/dist-packages/torch/backends/cuda/__init__.py:131: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return torch._C._get_cublas_allow_tf32()
W0130 10:41:19.826000 1158 torch/_inductor/utils.py:1558] [0/0] Not enough SMs to use max_autotune_gemm mode


Epoch 1 train {'loss': 0.8572380745894889, 'accuracy': 0.7283486912393162, 'gradients': {'total_l2_norm': 2.5842965477770616, 'per_layer_l2_norms': {'_orig_mod.conv1.weight': 2.080138683319092, '_orig_mod.conv1.bias': 3.632904918049462e-05, '_orig_mod.bn1.weight': 0.08771974593400955, '_orig_mod.bn1.bias': 0.0677151158452034, '_orig_mod.conv2.weight': 1.3400468826293945, '_orig_mod.conv2.bias': 1.1667250873870216e-05, '_orig_mod.bn2.weight': 0.10063015669584274, '_orig_mod.bn2.bias': 0.08189708739519119, '_orig_mod.conv3.weight': 0.46716007590293884, '_orig_mod.conv3.bias': 3.3332146358588943e-06, '_orig_mod.bn3.weight': 0.05420028418302536, '_orig_mod.bn3.bias': 0.06395591795444489, '_orig_mod.fc.weight': 0.5428256988525391, '_orig_mod.fc.bias': 0.0827411487698555}, 'zero_grad_parameters': 0}} val {'loss': 0.6138017841339112, 'accuracy': 0.7814}
Epoch 2 train {'loss': 0.5321394227381445, 'accuracy': 0.810012686965812, 'gradients': {'total_l2_norm': 2.64823362710488, 'per_layer_l2_norm